In [1]:
import os
import gc
import random
import numpy as np
import pandas as pd

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)



In [2]:
MIN_MAX = {
    "Weeks": (-5.0, 133.0),
    "FVC": (827.0, 6399.0),
    "Percent": (28.877577, 153.145378),
    "Age": (49.0, 88.0),
    "typical_fvc": (827.0, 6399.0),
}



In [3]:
IMG_SIZE = 48
NUM_OF_SCANS = 48
BATCH_SIZE = 64

DATA_DIR = "/kaggle/input/osic-pulmonary-fibrosis-progression"
TEST_DF = pd.read_csv(f"{DATA_DIR}/test.csv")
TRAIN_DF = pd.read_csv(f"{DATA_DIR}/train.csv")
SAMPLE_SUBMISSION = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

model_path = "../input/linear-model-osic-v2"
if os.path.isdir(model_path):
    model_weights = [os.path.join(model_path, x) for x in os.listdir(model_path)]
    model_weights = [w for w in model_weights if os.path.isfile(w)]
else:
    model_weights = []

training_features = [
    "Weeks",
    "min_week",
    "min_week_FVC",
    "typical_fvc",
    "Age",
    "Male",
    "Female",
    "Never smoked",
    "Currently smokes",
    "Ex-smoker",
]
num_of_features = len(training_features)




In [4]:
def create_typical_fvc(df):
    df = df.copy()
    df["typical_fvc"] = df["FVC"] / df["Percent"] * 100.0
    return df


def normalize(df):
    df = df.copy()
    for feature, min_max in MIN_MAX.items():
        if feature in df.columns:
            df[feature] = (df[feature] - min_max[0]) / (min_max[1] - min_max[0])
    return df


# NOTE: We keep your normalized TEST_DF for compatibility with the existing pipeline,
# but we will also keep an unnormalized copy for meaningful week/FVC computations.
TEST_DF_RAW = create_typical_fvc(TEST_DF)
TRAIN_DF_RAW = create_typical_fvc(TRAIN_DF)

TEST_DF = create_typical_fvc(TEST_DF)

TEST_DF["min_week"] = 0.0
TEST_DF["min_week_FVC"] = 0.0

TEST_DF["Never smoked"] = (TEST_DF["SmokingStatus"] == "Never smoked").astype("uint8")
TEST_DF["Currently smokes"] = (TEST_DF["SmokingStatus"] == "Currently smokes").astype(
    "uint8"
)
TEST_DF["Ex-smoker"] = (TEST_DF["SmokingStatus"] == "Ex-smoker").astype("uint8")

TEST_DF["Male"] = (TEST_DF["Sex"] == "Male").astype("uint8")
TEST_DF["Female"] = (TEST_DF["Sex"] == "Female").astype("uint8")

TEST_DF = normalize(TEST_DF)

for patient in np.unique(TEST_DF["Patient"]):
    mask = TEST_DF["Patient"] == patient
    TEST_DF.loc[mask, "min_week"] = float(TEST_DF.loc[mask, "Weeks"].min())
    TEST_DF.loc[mask, "min_week_FVC"] = float(TEST_DF.loc[mask, "FVC"].values[0])

TEST_DF.head()




,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,typical_fvc,min_week,min_week_FVC,Never smoked,Currently smokes,Ex-smoker,Male,Female
0,ID00014637202177757139317,0.036232,0.534817,0.492477,0.179487,Male,Ex-smoker,0.610086,0.036232,0.534817,0,0,1,1,0
1,ID00019637202178323708467,0.130435,0.228464,0.514865,0.871795,Female,Ex-smoker,0.257448,0.130435,0.228464,0,0,1,0,1
2,ID00047637202184938901501,0.050725,0.446159,0.491293,0.487179,Male,Ex-smoker,0.512742,0.050725,0.446159,0,0,1,1,0
3,ID00082637202201836229724,0.173913,0.375269,0.570681,0.000000,Female,Currently smokes,0.376346,0.173913,0.375269,0,1,0,0,1
4,ID00126637202218610655908,0.166667,0.277818,0.245417,0.743590,Male,Ex-smoker,0.569454,0.166667,0.277818,0,0,1,1,0


In [5]:
def _patient_seed(s: str) -> int:
    return (abs(hash(s)) + 12345) % (2**31 - 1)


def make_volume_from_tabular(patient_id: str, df_patient_row: pd.Series) -> np.ndarray:
    """
    Create a pseudo-CT volume in uint8 with shape (NUM_OF_SCANS, IMG_SIZE, IMG_SIZE),
    deterministically derived from baseline tabular features.
    """
    rng = np.random.RandomState(_patient_seed(patient_id))

    age = float(df_patient_row.get("Age", 0.5))
    fvc = float(df_patient_row.get("FVC", 0.5))
    percent = float(df_patient_row.get("Percent", 0.5))
    typical = float(df_patient_row.get("typical_fvc", 0.5))

    mu = 40.0 + 120.0 * np.clip(
        0.35 * typical + 0.25 * percent + 0.25 * fvc + 0.15 * age, 0.0, 1.0
    )
    mu = float(np.clip(mu, 0.0, 255.0))

    vol = rng.normal(
        loc=mu, scale=18.0, size=(NUM_OF_SCANS, IMG_SIZE, IMG_SIZE)
    ).astype(np.float32)
    z = np.linspace(-1.0, 1.0, NUM_OF_SCANS, dtype=np.float32)[:, None, None]
    vol += z * 10.0

    vol = np.clip(vol, 0.0, 255.0).astype(np.uint8)
    return vol


volumes = {}
for patient_id in np.unique(TEST_DF["Patient"]):
    row = TEST_DF[TEST_DF["Patient"] == patient_id].iloc[0]
    volumes[patient_id] = make_volume_from_tabular(patient_id, row)

gc.collect()




0

In [6]:
class Dataset:
    def __init__(self, batch_size=BATCH_SIZE, mode=0):
        self.indices = np.arange(0, len(SAMPLE_SUBMISSION), 1)
        self.batch_size = batch_size
        self.mode = mode  # 0 - Training, 1 - Test

    def __len__(self):
        return int(np.ceil(len(self.indices) / float(self.batch_size)))

    def get_tabular(self, patient, week):
        week = (float(week) - MIN_MAX["Weeks"][0]) / (
            MIN_MAX["Weeks"][1] - MIN_MAX["Weeks"][0]
        )
        tabular = [week]
        tabular += list(
            TEST_DF[training_features[1:]][TEST_DF["Patient"] == patient].values[0]
        )
        return np.asarray(tabular, dtype="float32")

    def get_volume(self, patient_id):
        return volumes[patient_id]

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min((index + 1) * self.batch_size, len(self.indices))
        indices = self.indices[start:end]

        Patient_Week = np.asarray(SAMPLE_SUBMISSION["Patient_Week"][indices])
        patient = [x.split("_")[0] for x in Patient_Week]
        week = [x.split("_")[1] for x in Patient_Week]

        images = (
            np.asarray(
                [self.get_volume(patient_id) for patient_id in patient],
                dtype=np.float32,
            )
            / 255.0
        )
        images = np.expand_dims(images, axis=4)  # (B, D, H, W, 1)
        tabulars = np.asarray(
            [self.get_tabular(patient[i], week[i]) for i in range(len(patient))],
            dtype="float32",
        )
        return [images, tabulars]




In [7]:
models = []
len(models), 0



(0, 0)

In [8]:
test_gen = Dataset(mode=1)
predictions = None



In [9]:
# Change (score improvement): learn a robust "typical decline slope" from TRAIN_DF_RAW and apply
# a personalized linear trend per test patient instead of constant baseline FVC.
# This keeps the core deterministic/tabular fallback approach, but fixes the main source of error:
# predicting identical FVC across weeks.


def robust_patient_slope(df_p: pd.DataFrame) -> float:
    """Return slope (ml/week) from least-squares on (Weeks, FVC)."""
    x = df_p["Weeks"].values.astype(np.float64)
    y = df_p["FVC"].values.astype(np.float64)
    if len(x) < 2:
        return np.nan
    x0 = x.mean()
    y0 = y.mean()
    denom = np.sum((x - x0) ** 2)
    if denom <= 1e-12:
        return np.nan
    return float(np.sum((x - x0) * (y - y0)) / denom)


# Compute per-patient slopes from training and take a robust central tendency.
train_slopes = []
for pid, df_p in TRAIN_DF_RAW.groupby("Patient"):
    s = robust_patient_slope(df_p)
    if np.isfinite(s):
        # Clip extreme slopes to avoid overfitting/outlier influence.
        train_slopes.append(np.clip(s, -30.0, 10.0))

train_slopes = np.asarray(train_slopes, dtype=np.float64)
global_slope = float(np.median(train_slopes)) if len(train_slopes) else -8.0  # ml/week

# Estimate baseline week per test patient (available point in test.csv).
test_base_week = TEST_DF_RAW.set_index("Patient")["Weeks"].to_dict()
test_base_fvc = TEST_DF_RAW.set_index("Patient")["FVC"].to_dict()
test_base_percent = TEST_DF_RAW.set_index("Patient")["Percent"].to_dict()

# Parse requested Patient_Week rows
pw = SAMPLE_SUBMISSION["Patient_Week"].values
patients = np.array([x.split("_")[0] for x in pw])
weeks_req = np.array([float(x.split("_")[1]) for x in pw], dtype=np.float32)

base_week_arr = np.array([test_base_week[p] for p in patients], dtype=np.float32)
base_fvc_arr = np.array([test_base_fvc[p] for p in patients], dtype=np.float32)
base_percent_arr = np.array([test_base_percent[p] for p in patients], dtype=np.float32)

# Personalize slope mildly by baseline Percent: lower Percent -> slightly faster decline.
# Keep effect small to avoid destabilizing; slope remains near global median.
percent_center = float(np.median(TRAIN_DF_RAW["Percent"].values))
percent_scale = 40.0
slope_person = global_slope * (
    1.0 + 0.15 * np.clip((percent_center - base_percent_arr) / percent_scale, -1.0, 1.0)
)
slope_person = slope_person.astype(np.float32)

dt = (weeks_req - base_week_arr).astype(np.float32)
FVC_pred = base_fvc_arr + slope_person * dt

# Keep predictions within plausible range to reduce large deltas; this helps the metric (delta capped at 1000).
FVC_pred = np.clip(FVC_pred, MIN_MAX["FVC"][0], MIN_MAX["FVC"][1]).astype(np.float32)

# Change (score improvement): confidence should grow with temporal distance.
# Use a simple calibrated form; always >= 70 due to metric clip.
abs_dt = np.abs(dt).astype(np.float32)
Confidence = (90.0 + 2.0 * abs_dt).astype(np.float32)
Confidence = np.clip(Confidence, 70.0, 500.0)

SAMPLE_SUBMISSION["FVC"] = np.round(FVC_pred).astype(int)
SAMPLE_SUBMISSION["Confidence"] = np.round(Confidence).astype(int)
SAMPLE_SUBMISSION.to_csv("submission.csv", index=False)
SAMPLE_SUBMISSION.head()

,Patient_Week,FVC,Confidence
0,ID00126637202218610655908_-3,2456,132
1,ID00126637202218610655908_-2,2452,130
2,ID00126637202218610655908_-1,2448,128
3,ID00126637202218610655908_0,2444,126
4,ID00126637202218610655908_1,2440,124
